In [2]:
import pandas as pd
import requests
import pandas as pd
import time
import json
import os
import sys
import pandas as pd
import re
from datetime import datetime
import pandas as pd
import numpy as np
import re


def map_action_numbers(old_df, new_df):
    """
    Map actionNumber from new_df to the appropriate rows in old_df based on time ranges and periods.
    
    Parameters:
    old_df (pandas.DataFrame): The old dataset with start_seconds and end_seconds columns
    new_df (pandas.DataFrame): The new dataset with actionNumber, period, and clock columns
    
    Returns:
    pandas.DataFrame: A copy of old_df with a new column 'actionNumber' added
    """
    # Make a copy of the old_df to avoid modifying the original
    result_df = old_df.copy()
    
    # Initialize the actionNumber column with NaN
    result_df['actionNumber'] = np.nan
    
    # Convert data types to ensure proper comparison
    result_df['PERIOD'] = result_df['PERIOD'].astype(int)
    result_df['start_seconds'] = result_df['start_seconds'].astype(float)
    result_df['end_seconds'] = result_df['end_seconds'].astype(float)
    
    new_df['period'] = new_df['period'].astype(int)
    new_df['actionNumber'] = new_df['actionNumber'].astype(int)
    
    # Helper function to convert clock format (PT12M00.00S) to seconds
    def clock_to_seconds(clock_str):
        if pd.isna(clock_str) or not isinstance(clock_str, str):
            return None
            
        # Extract minutes and seconds using regex
        minutes_match = re.search(r'PT(\d+)M', clock_str)
        seconds_match = re.search(r'M(\d+\.\d+)S', clock_str)
        
        if not seconds_match:
            seconds_match = re.search(r'M(\d+)S', clock_str)
            
        minutes = int(minutes_match.group(1)) if minutes_match else 0
        seconds = float(seconds_match.group(1)) if seconds_match else 0
        
        return minutes * 60 + seconds
    
    # Calculate game seconds for each action in the new dataset
    new_df['clock_seconds'] = new_df['clock'].apply(clock_to_seconds)
    new_df['period_start_seconds'] = (new_df['period'] - 1) * 720
    new_df['seconds_into_period'] = 720 - new_df['clock_seconds']
    new_df['game_seconds'] = new_df['period_start_seconds'] + new_df['seconds_into_period']
    
    # Sort new_df by period and game_seconds (should already be in order but just to be sure)
    new_df = new_df.sort_values(['period', 'game_seconds'])
    
    # Group by period for faster access
    new_df_by_period = {period: group for period, group in new_df.groupby('period')}
    
    # Function to find the nearest action number for a given time range and period
    def find_action_number(row):
        period = row['PERIOD']
        start_time = row['start_seconds']
        end_time = row['end_seconds']
        
        # Check if we have data for this period
        if period not in new_df_by_period:
            return np.nan
        
        period_data = new_df_by_period[period]
        
        # Find actions that fall within the time range
        matches = period_data[
            (period_data['game_seconds'] >= start_time) & 
            (period_data['game_seconds'] <= end_time)
        ]
        
        if not matches.empty:
            # Return the first action number in the time range
            # (actions are already ordered chronologically)
            return matches['actionNumber'].iloc[0]
        
        # If no direct match, find the closest action before the time range
        before_matches = period_data[period_data['game_seconds'] < start_time]
        if not before_matches.empty:
            return before_matches['actionNumber'].iloc[-1]
        
        # If still no match, find the closest action after the time range
        after_matches = period_data[period_data['game_seconds'] > end_time]
        if not after_matches.empty:
            return after_matches['actionNumber'].iloc[0]
        
        return np.nan
    
    # Apply the function to each row in the old dataset
    result_df['actionNumber'] = result_df.apply(find_action_number, axis=1)
    
    return result_df


def main():
    # Example usage
    old_df = pd.read_csv('old.csv')
    new_df = pd.read_csv('new.csv')
    
    result_df = map_action_numbers(old_df, new_df)
    
    # Save the result
    result_df.to_csv('mapped_results.csv', index=False)
    print(f"Mapping complete. Result saved to 'mapped_results.csv'")
    
    # Display some stats
    total_rows = len(result_df)
    mapped_rows = result_df['actionNumber'].notna().sum()
    mapping_percentage = (mapped_rows / total_rows) * 100
    
    print(f"Total rows in old dataset: {total_rows}")
    print(f"Successfully mapped rows: {mapped_rows} ({mapping_percentage:.2f}%)")


def pull_data(url):
    headers = {
        "Host": "stats.nba.com",
        "User-Agent": "Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/116.0.0.0 Safari/537.36",
        "Accept": "application/json, text/plain, */*",
        "Accept-Language": "en-US,en;q=0.9",
        "Accept-Encoding": "gzip, deflate, br",
        "Connection": "keep-alive",
        "Referer": "https://stats.nba.com/",
        "Origin": "https://stats.nba.com",
        "Sec-Fetch-Dest": "empty",
        "Sec-Fetch-Mode": "cors",
        "Sec-Fetch-Site": "same-origin",
    }

    response = requests.get(url, headers=headers)
    json = response.json()
    
    # Handle the video events format
    if 'resultSets' in json and isinstance(json['resultSets'], dict):
        if 'Meta' in json['resultSets'] and 'videoUrls' in json['resultSets']['Meta']:
            video_urls = json['resultSets']['Meta']['videoUrls']
            playlist = json['resultSets'].get('playlist', [])
            
            # Convert video URLs to dataframe
            video_df = pd.DataFrame(video_urls)
            
            # Convert playlist to dataframe
            playlist_df = pd.DataFrame(playlist)
            
            # Merge video and playlist data
            if not playlist_df.empty:
                df = pd.concat([video_df, playlist_df], axis=1)
            else:
                df = video_df

        else:
            # Fallback in case no video data is found
            df = pd.DataFrame()

    # Handle the original stats format
    elif 'resultSets' in json and isinstance(json['resultSets'], list):
        if len(json["resultSets"]) == 1:
            data = json["resultSets"][0]["rowSet"]
            columns = json["resultSets"][0]["headers"]
            df = pd.DataFrame.from_records(data, columns=columns)
        else:
            data = json["resultSets"][1]["rowSet"]
            columns = json["resultSets"][1]["headers"]["columnNames"]
            df = pd.DataFrame.from_records(data, columns=columns)

    else:
        # Empty dataframe if no recognizable format is found
        df = pd.DataFrame()

    time.sleep(1.2)
    return df

import pandas as pd



# List of NBA team acronyms


# Example: Access the DataFrame for the Atlanta Hawks
# atl_df = team_dfs['ATL']

result_frames=[]
teams = [
    'ATL', 'BOS', 'BKN', 'CHA', 'CHI', 'CLE', 'DAL', 'DEN', 'DET', 'GSW', 
    'HOU', 'IND', 'LAC', 'LAL', 'MEM', 'MIA', 'MIL', 'MIN', 'NOP', 'NYK', 
    'OKC', 'ORL', 'PHI', 'PHX', 'POR', 'SAC', 'SAS', 'TOR', 'UTA', 'WAS'
]

# Dictionary to store DataFrames for each team
team_dfs = {}

# Loop through each team and read the CSV file
for team in teams:
    file_path = f'2025/{team}_2025_clips_with_players.csv'
    
    if os.path.exists(file_path):  # Ensure the file exists before reading
        team_dfs[team] = pd.read_csv(file_path)
        print(f"Loaded {team}")
    else:
        print(f"File not found: {file_path}")


    all_df = pd.read_csv(file_path)

    # Identify GAME_IDs with at least one non-NaN URL
    valid_games = all_df[all_df['URL'].notna()]['GAMEID'].unique()

    # Filter out GAME_IDs without any non-NaN URLs
    missing_url_games = all_df[~all_df['GAMEID'].isin(valid_games)]['GAMEID'].unique()

    print("GAME_IDs without at least one non-NaN URL:")
    print(missing_url_games)


    missing=all_df[all_df.GAMEID.isin(missing_url_games)]
    missing

    # API endpoint

    all_rows = []
    for game_id in missing_url_games:
  


    # Direct API endpoint for play-by-play data

        url = f"https://cdn.nba.com/static/json/liveData/playbyplay/playbyplay_00{game_id}.json"

        # Set headers to mimic a browser request
        headers = {
            "User-Agent": "Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/122.0.0.0 Safari/537.36"
        }

        # Fetch the JSON data
        response = requests.get(url, headers=headers)


        if response.status_code == 200:
            data = response.json()

            actions = data.get('game', {}).get('actions', [])
                
                # Convert each action into a dictionary and add to the list
            for action in actions:
                action['game_id'] = game_id  # Add the game ID as a column
                all_rows.append(action)
            
        else:
            print(f"Failed to fetch data: {response.status_code}")
        time.sleep(1)
    time.sleep(1)
    teamdf = pd.DataFrame(all_rows)


    old_df=missing.copy()

    new_df=teamdf.copy()

    # Example usage
    old_df['GAMEID']='00'+old_df['GAMEID'].astype(str)
    old_df.sort_values(by='GAMEDATE',inplace=True)
    new_df.sort_values(by='timeActual',inplace=True)
    teamid=old_df['TEAM_ID'].iloc[0]
    new_df=new_df[new_df.teamId==teamid]
    result_df = map_action_numbers(old_df, new_df)
    
    # Save the result
    result_df.to_csv('mapped_results.csv', index=False)
    print(f"Mapping complete. Result saved to 'mapped_results.csv'")
    
    # Display some stats
    total_rows = len(result_df)
    mapped_rows = result_df['actionNumber'].notna().sum()
    mapping_percentage = (mapped_rows / total_rows) * 100
    
    print(f"Total rows in old dataset: {total_rows}")
    print(f"Successfully mapped rows: {mapped_rows} ({mapping_percentage:.2f}%)")

    result_frames.append(result_df)
data=pd.concat(result_frames)
data

Loaded ATL
GAME_IDs without at least one non-NaN URL:
[22400239 22400719 22400945 22400960 22400978 22400993]
Mapping complete. Result saved to 'mapped_results.csv'
Total rows in old dataset: 1193
Successfully mapped rows: 1193 (100.00%)
Loaded BOS
GAME_IDs without at least one non-NaN URL:
[22400958 22400968 22400994]
Mapping complete. Result saved to 'mapped_results.csv'
Total rows in old dataset: 532
Successfully mapped rows: 532 (100.00%)
Loaded BKN
GAME_IDs without at least one non-NaN URL:
[22400720 22400956 22400968 22400978 22400994 22401011]
Mapping complete. Result saved to 'mapped_results.csv'
Total rows in old dataset: 1170
Successfully mapped rows: 1170 (100.00%)
Loaded CHA
GAME_IDs without at least one non-NaN URL:
[22400538 22400717 22400945 22400964 22400993 22401010]
Mapping complete. Result saved to 'mapped_results.csv'
Total rows in old dataset: 1115
Successfully mapped rows: 1115 (100.00%)
Loaded CHI
GAME_IDs without at least one non-NaN URL:
[22400723 22400956 2240

,index,ENDTIME,EVENTS,FG2A,FG2M,FG3A,FG3M,GAMEDATE,GAMEID,NONSHOOTINGFOULSTHATRESULTEDINFTS,...,team,TEAM_ID,Year,start_seconds,end_seconds,mid_seconds,players_on,opp_players_on,season,actionNumber
3089,111330,01:55,Krejčí 5' Driving Dunk (5 PTS) (Capela 3 AST)\n,1,1,0,0,2024-11-17,0022400239,0,...,ATL,1610612737,2024,1320.0,1325.0,1322.5,203991|1629027|1630249|1630552|1630700,1630166|1630625|1630703|1641739|1642270,2024-25,300
3221,111461,08:22,Henderson BLOCK (1 BLK): MISS Okongwu 2' Cutti...,1,0,0,0,2024-11-17,0022400239,0,...,ATL,1610612737,2024,2368.0,2378.0,2373.0,1629027|1630168|1630552|1630700|1630811,1630166|1630703|1631101|1641739|1642270,2024-25,608
3222,111416,09:32,Grant P.FOUL (P4.T1) (B.Forte)\nHenderson S.FO...,0,0,0,0,2024-11-17,0022400239,0,...,ATL,1610612737,2024,1575.0,1588.0,1581.5,203991|1629027|1630552|1630700|1642258,203924|1630703|1631101|1641739|1642270,2024-25,404
3223,111460,08:22,Henderson BLOCK (1 BLK): MISS Okongwu 2' Cutti...,1,0,0,0,2024-11-17,0022400239,0,...,ATL,1610612737,2024,2368.0,2378.0,2373.0,1629027|1630168|1630552|1630700|1630811,1630166|1630703|1631101|1641739|1642270,2024-25,608
3224,111458,07:56,MISS Daniels 25' 3PT Pullup Jump Shot\nClingan...,0,0,1,0,2024-11-17,0022400239,0,...,ATL,1610612737,2024,2379.0,2404.0,2391.5,1629027|1630168|1630552|1630700|1630811,1630166|1630703|1631101|1641739|1642270,2024-25,547
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
7999,394778,02:04,George P.FOUL (P2.T3) (N.Buchert)\nGill 3' Lay...,1,1,0,0,2025-03-19,0022400627,0,...,WAS,1610612764,2025,2728.0,2756.0,2742.0,1630264|1630550|1641732|1641798|1642358,1630548|1641718|1642262|1642268|1642271,2024-25,685
7998,394779,02:04,George P.FOUL (P2.T3) (N.Buchert)\nGill 3' Lay...,1,1,0,0,2025-03-19,0022400627,0,...,WAS,1610612764,2025,2728.0,2756.0,2742.0,1630264|1630550|1641732|1641798|1642358,1630548|1641718|1642262|1642268|1642271,2024-25,685
7997,394780,02:48,Gill OFF.Foul (P3) (N.Buchert)\nGill Offensive...,0,0,0,0,2025-03-19,0022400627,0,...,WAS,1610612764,2025,2695.0,2712.0,2703.5,1630264|1630550|1641732|1641798|1642358,1630548|1641718|1642262|1642268|1642271,2024-25,622
7995,394782,03:19,Thor 3' Cutting Layup Shot (8 PTS) (Gill 2 AST)\n,1,1,0,0,2025-03-19,0022400627,0,...,WAS,1610612764,2025,2659.0,2681.0,2670.0,1630264|1630550|1641732|1641798|1642358,1630548|1641718|1642262|1642268|1642271,2024-25,587


In [3]:
data.sort_values(by=['GAMEDATE','GAMEID','PERIOD','start_seconds'],inplace=True)
data.to_csv('missing_actions.csv',index=False)
data['GAMEID'] = data['GAMEID'].str.replace(r'^00', '', regex=True)
data['GAMEID']= data['GAMEID'].astype(int)

In [ ]:
import pandas as pd
action_map = pd.read_csv('nba_ping_results_final.csv')
action_map




action_map.rename(columns={'game_id':'GAMEID','action_number':'actionNumber'},inplace=True)
action_map=action_map[['GAMEID','actionNumber','url']]



newdata= data.merge(action_map,how='left')
newdata.rename(columns={'url':'URL'},inplace = True)





teams = [
    'ATL', 'BOS', 'BKN', 'CHA', 'CHI', 'CLE', 'DAL', 'DEN', 'DET', 'GSW', 
    'HOU', 'IND', 'LAC', 'LAL', 'MEM', 'MIA', 'MIL', 'MIN', 'NOP', 'NYK', 
    'OKC', 'ORL', 'PHI', 'PHX', 'POR', 'SAC', 'SAS', 'TOR', 'UTA', 'WAS'
]

# Dictionary to store DataFrames for each team
team_dfs = {}

# Remove leading '00' from each game ID
#newdata['GAMEID'] = newdata['GAMEID'].str.lstrip('0').astype(int)
print(newdata['GAMEID'])
newdata['GAMEID']=newdata['GAMEID'].astype(int)
print(newdata['GAMEID'])
# Loop through each team and read the CSV file
for team in teams:
    file_path = f'2025/{team}_2025_clips_with_players.csv'
    df= pd.read_csv(file_path)
    teamid=df['TEAM_ID'].iloc[0]
    newdata=newdata

    df=df[~df.GAMEID.isin(newdata)]
    print(len(df.GAMEID.unique()))
    # Remove duplicate columns by transposing, dropping duplicates, and transposing back
    teamnew = teamnew.loc[:, ~teamnew.columns.duplicated()].copy()

    # Verify the columns
    print(teamnew.columns)

    
    print(len(teamnew.GAMEID.unique()))
    df = pd.concat([df,teamnew])
    df.sort_values(by=['GAMEDATE','PERIOD','start_seconds'],inplace=True)
    df.drop(columns=['level_0'])
    df.to_csv(file_path,index=False)
    print(len(df.GAMEID.unique()))
 

0        22400239
1        22400239
2        22400239
3        22400239
4        22400239
           ...   
31906    22401013
31907    22401013
31908    22401013
31909    22401013
31910    22401013
Name: GAMEID, Length: 31911, dtype: int64
0        22400239
1        22400239
2        22400239
3        22400239
4        22400239
           ...   
31906    22401013
31907    22401013
31908    22401013
31909    22401013
31910    22401013
Name: GAMEID, Length: 31911, dtype: int64
68
Index(['level_0', 'index', 'ENDTIME', 'EVENTS', 'FG2A', 'FG2M', 'FG3A', 'FG3M',
       'GAMEDATE', 'GAMEID', 'NONSHOOTINGFOULSTHATRESULTEDINFTS',
       'OFFENSIVEREBOUNDS', 'OPPONENT', 'PERIOD', 'SHOOTINGFOULSDRAWN',
       'STARTSCOREDIFFERENTIAL', 'STARTTIME', 'STARTTYPE', 'TURNOVERS',
       'DESCRIPTION', 'URL', 'HTM', 'VTM', 'team', 'TEAM_ID', 'Year',
       'start_seconds', 'end_seconds', 'mid_seconds', 'players_on',
       'opp_players_on', 'season', 'actionNumber'],
      dtype='object')
6
68
69
Index([